# Uncertainty Quantification (Phase 7)

For every dataset and every shared output target, this notebook computes out-of-fold Prediction Intervals (PI) for GPR plus the Top-3 models:

* **GPR (Kriging)** — analytical PI from the Gaussian Process posterior predictive std.
* **All other models (Top-3)** — percentile Bootstrap PI (resample the training fold, refit, collect the prediction distribution; `configs/config.yaml: uncertainty.bootstrap_iterations` resamples per fold).

Reported per (dataset, target, model): mean PI width and empirical coverage (fraction of true values inside the interval at the configured confidence level). These numbers also feed the Top-3 multi-criteria selection's uncertainty criterion.

Note: with a 95% confidence level and `bootstrap_iterations=200` refits per fold, the bootstrap models are the slowest part of this notebook — this can take a while to complete for all dataset/target/model combinations.

Setup: repo imports, pandas display options, load the Top-3 models and all datasets.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
SCRIPTS_DIR = REPO_ROOT / 'scripts'
for p in (REPO_ROOT, SCRIPTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from _pipeline_common import read_top3_models
from src.config import get_base_seed, get_path, load_config
from src.data.loader import load_all_datasets
from src.evaluation import bootstrap_prediction_interval, gpr_prediction_interval, interval_metrics
from src.models.registry import build_model
from src.utils.io import save_table

cfg = load_config()
seed = get_base_seed()
n_splits = cfg['cross_validation']['n_splits']
models = sorted(set(read_top3_models()) | {'gpr'})
bundles = load_all_datasets(discrete=False)
print('Models evaluated (Top-3 union GPR):', models)
print('Confidence level:', cfg['uncertainty']['confidence_level'])
print('Bootstrap iterations per fold:', cfg['uncertainty']['bootstrap_iterations'])

## Prediction intervals on Dataset_0136

For every shared target and every model: 5-fold CV, GPR gets the analytical PI, everything else gets the bootstrap PI; predictions across folds are concatenated before computing PI width and coverage.

In [ ]:
rows_0136 = []
bundle = bundles['Dataset_0136']
X = bundle.X.to_numpy()
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target].to_numpy()
    for model_name in models:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y_true_all, lo_all, hi_all = [], [], []
        for tr, te in kf.split(X):
            pipe = build_model(model_name, seed=seed)
            pipe.fit(X[tr], y[tr])
            if model_name == 'gpr':
                _, lo, hi = gpr_prediction_interval(pipe, X[te])
            else:
                _, lo, hi = bootstrap_prediction_interval(
                    build_model(model_name, seed=seed), X[tr], y[tr], X[te], seed=seed
                )
            y_true_all.append(y[te]); lo_all.append(lo); hi_all.append(hi)
        metrics = interval_metrics(
            np.concatenate(y_true_all), np.concatenate(lo_all), np.concatenate(hi_all)
        )
        rows_0136.append({'dataset': 'Dataset_0136', 'target': target, 'model': model_name, **metrics})
        print(f'[uncertainty] Dataset_0136 / {target} / {model_name}: {metrics}')

uncertainty_0136 = pd.DataFrame(rows_0136)
uncertainty_0136

## Prediction intervals on Dataset_0172

For every shared target and every model: 5-fold CV, GPR gets the analytical PI, everything else gets the bootstrap PI; predictions across folds are concatenated before computing PI width and coverage.

In [ ]:
rows_0172 = []
bundle = bundles['Dataset_0172']
X = bundle.X.to_numpy()
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target].to_numpy()
    for model_name in models:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y_true_all, lo_all, hi_all = [], [], []
        for tr, te in kf.split(X):
            pipe = build_model(model_name, seed=seed)
            pipe.fit(X[tr], y[tr])
            if model_name == 'gpr':
                _, lo, hi = gpr_prediction_interval(pipe, X[te])
            else:
                _, lo, hi = bootstrap_prediction_interval(
                    build_model(model_name, seed=seed), X[tr], y[tr], X[te], seed=seed
                )
            y_true_all.append(y[te]); lo_all.append(lo); hi_all.append(hi)
        metrics = interval_metrics(
            np.concatenate(y_true_all), np.concatenate(lo_all), np.concatenate(hi_all)
        )
        rows_0172.append({'dataset': 'Dataset_0172', 'target': target, 'model': model_name, **metrics})
        print(f'[uncertainty] Dataset_0172 / {target} / {model_name}: {metrics}')

uncertainty_0172 = pd.DataFrame(rows_0172)
uncertainty_0172

## Prediction intervals on Dataset_3772

For every shared target and every model: 5-fold CV, GPR gets the analytical PI, everything else gets the bootstrap PI; predictions across folds are concatenated before computing PI width and coverage.

In [ ]:
rows_3772 = []
bundle = bundles['Dataset_3772']
X = bundle.X.to_numpy()
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target].to_numpy()
    for model_name in models:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y_true_all, lo_all, hi_all = [], [], []
        for tr, te in kf.split(X):
            pipe = build_model(model_name, seed=seed)
            pipe.fit(X[tr], y[tr])
            if model_name == 'gpr':
                _, lo, hi = gpr_prediction_interval(pipe, X[te])
            else:
                _, lo, hi = bootstrap_prediction_interval(
                    build_model(model_name, seed=seed), X[tr], y[tr], X[te], seed=seed
                )
            y_true_all.append(y[te]); lo_all.append(lo); hi_all.append(hi)
        metrics = interval_metrics(
            np.concatenate(y_true_all), np.concatenate(lo_all), np.concatenate(hi_all)
        )
        rows_3772.append({'dataset': 'Dataset_3772', 'target': target, 'model': model_name, **metrics})
        print(f'[uncertainty] Dataset_3772 / {target} / {model_name}: {metrics}')

uncertainty_3772 = pd.DataFrame(rows_3772)
uncertainty_3772

## Final Summary — Complete Prediction Interval Results (All Datasets)

PI width and coverage for every (dataset, target, model) combination, saved to `results/uncertainty/prediction_intervals.csv`.

In [ ]:
uncertainty_all = pd.concat([uncertainty_0136, uncertainty_0172, uncertainty_3772], ignore_index=True)
save_table(uncertainty_all, get_path('results_dir') / 'uncertainty' / 'prediction_intervals.csv')

print('=' * 90)
print('PREDICTION INTERVAL RESULTS — PI width and empirical coverage (all datasets)')
print('=' * 90)
display(uncertainty_all)

print()
print('Mean PI width and coverage by model (averaged over datasets/targets):')
display(uncertainty_all.groupby('model')[['pi_mean_width', 'pi_coverage']].mean())

print()
print('Uncertainty results saved to:', (get_path('results_dir') / 'uncertainty').resolve())